In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv('train.csv')

# Проверка, все ли поля есть в БД
needed_cols = ['Age', 'Sex', 'Pclass', 'Fare', 'Embarked', 'Cabin', 'Survived']
assert all(
    col in df.columns for col in needed_cols
), 'Не все поля есть!:('

# Ч2
n = int(input('Сколько первых строк вывести? '))
print(df.head(n))
print('\nПоследние 5 строк:')
print(df.tail())

print(f'\nРазмер таблицы: {df.shape[0]} строк, {df.shape[1]} столбцов')
print('Столбцы:', list(df.columns))
print('\nТипы данных:\n', df.dtypes)

# Разделяем признаки
num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
cat_cols = [
    'PassengerId',
    'Survived',
    'Pclass',
    'Name',
    'Sex',
    'Ticket',
    'Cabin',
    'Embarked',
]

print('\nСтатистика по числовым признакам:')
print(df[num_cols].agg(['mean', 'median', 'min', 'max', 'std']).round(2))

print('\nПассажиров по классам:')
print(df['Pclass'].value_counts().sort_index())

print(f"\nУникальных портов посадки, без NaNов: {df['Embarked'].nunique()}")


#Ч3
print('\nПропуски по столбцам:')
print(df.isna().sum())

print('\nПроцент пропусков:')
print((df.isna().mean() * 100).round(2))

max_na_col = df.isna().sum().idxmax()
print(f'\nБольше всего пропусков в столбце: {max_na_col}')


# Считаем статистику по возрасту после итзменения
def age_summary(s):
  return [
      round(s.mean(), 2),
      round(s.median(), 2),
      s.min(),
      s.max(),
      round(s.std(), 2),
      s.count(),
  ]


rows_before, cols_before = df.shape
before_stats = age_summary(df['Age'])

# Заполняем пропуски в Age(медианой)
df['Age'] = df['Age'].fillna(df['Age'].median())
after_stats = age_summary(df['Age'])

# Собираем таблицу сравнения
metrics = [
    'Среднее',
    'Медиана',
    'Минимум',
    'Максимум',
    'Ст. отклонение',
    'Известных значений',
]
age_comp = pd.DataFrame(
    {'До': before_stats, 'После': after_stats}, index=metrics
)
print('\nСравнение Age до и после:')
print(age_comp)

# Удаляем строки без Embarked
df = df.dropna(subset=['Embarked'])

# удаляем столбец, если более 50 % пропусков
if df['Cabin'].isna().mean() > 0.5:
  df = df.drop(columns=['Cabin'])
  print('\nСтолбец Cabin удалён, тк слишком много пропусков).')

print('\nОставшиеся пропуски:\n', df.isna().sum())
print(
    f'Было: {rows_before}x{cols_before}, стало: {df.shape[0]}x{df.shape[1]}\n'
)


# Ч4
survived = df['Survived'].to_numpy()
sex = df['Sex'].to_numpy()
age = df['Age'].to_numpy()
fare = df['Fare'].to_numpy()

is_male = sex == 'male'
is_female = sex == 'female'

print(f'Общий процент выживших: {np.mean(survived) * 100:.2f}%')
print(
    f'Билеты: мин = {np.min(fare):.2f}, макс = {np.max(fare):.2f}, среднее ='
    f' {np.mean(fare):.2f}'
)
print(f'Ст. отклонение возраста: {np.std(age, ddof=1):.2f}')

# Таблица по полу(после анализа)
gender_table = pd.DataFrame({
    'Показатель': [
        'Выжившие (%)',
        'Средний возраст',
        'Возраст выживших',
        'Возраст погибших',
    ],
    'Мужчины': [
        round(np.mean(survived[is_male]) * 100, 2),
        round(np.mean(age[is_male]), 2),
        round(np.mean(age[is_male & (survived == 1)]), 2),
        round(np.mean(age[is_male & (survived == 0)]), 2),
    ],
    'Женщины': [
        round(np.mean(survived[is_female]) * 100, 2),
        round(np.mean(age[is_female]), 2),
        round(np.mean(age[is_female & (survived == 1)]), 2),
        round(np.mean(age[is_female & (survived == 0)]), 2),
    ],
})
print('\nАнализ по полу (NumPy):')
print(gender_table.to_string(index=False))


# Ч5
# Мужчины > 30 лет в 1 классе
f1 = df[(df['Sex'] == 'male') & (df['Age'] > 30) & (df['Pclass'] == 1)]
print(f'\n1. Мужчины > 30 лет, 1 класс: {len(f1)} чел.')
print(f1.head(10))

# Дети (те меньше 18 лет) или женщины, которые выжили
f2 = df[((df['Age'] < 18) | (df['Sex'] == 'female')) & (df['Survived'] == 1)]
print(f'\n2. Выжившие дети или женщины: {len(f2)} чел.')

# 3-й класс с билетом дороже среднего по их классу
mean_fare_3 = df[df['Pclass'] == 3]['Fare'].mean()
f3 = df[(df['Pclass'] == 3) & (df['Fare'] > mean_fare_3)]
print(f'3. 3-й класс с билетом дороже среднего ({mean_fare_3:.2f}): {len(f3)} чел.')

# Пассажиры 20-40 лет из порта S
f4 = df[(df['Age'].between(20, 40)) & (df['Embarked'] == 'S')]
print(f'4. Возраст 20–40 лет из порта S: {len(f4)} чел.')


# Ч6
stats_pclass_sex = (
    df.groupby(['Pclass', 'Sex'])
    .agg(
        count=('Survived', 'count'),
        mean_age=('Age', 'mean'),
        survival_rate=('Survived', 'mean'),
        mean_fare=('Fare', 'mean'),
    )
    .reset_index()
)

# Переводим долю выживших в проценты,далее округляем
stats_pclass_sex['survival_rate'] = (
    stats_pclass_sex['survival_rate'] * 100
).round(2)
stats_pclass_sex[['mean_age', 'mean_fare']] = stats_pclass_sex[
    ['mean_age', 'mean_fare']
].round(2)

print('\nГруппировка по классу и полу:')
print(stats_pclass_sex.to_string(index=False))

stats_embarked = (
    df.groupby('Embarked')
    .agg(count=('Survived', 'count'), survival_rate=('Survived', 'mean'))
    .reset_index()
)
stats_embarked['survival_rate'] = (stats_embarked['survival_rate'] * 100).round(
    2
)

print('\nГруппировка по порту:')
print(stats_embarked.to_string(index=False))

top_class = df.groupby('Pclass')['Survived'].mean().idxmax()
print(f'\nБольше всего выживших в {top_class}-м классе.')


# Ч7
df.to_csv('titanic_processed.csv', index=False)
stats_pclass_sex.to_csv('titanic_statistics.csv', index=False)

# Проверяем, что всё сохранилось нормально
check = pd.read_csv('titanic_processed.csv')
print(f'\nПроверка сохранения: загружено {len(check)} строк.')


feerfgerge
